# Gemini Image Analysis Setup

In this section, we import the necessary libraries, set up our project path to access local utilities, and initialize the Gemini client.

In [9]:
# Load env variables and create client
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from PIL import Image

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "requirements.txt").exists()),
    Path.cwd(),
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.gemini_retry import generate_content_with_retry

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = "gemini-2.5-flash"

## Helper Functions

These functions simplify message management and interaction with the Gemini API, including built-in retry logic for rate limits.

In [10]:
# Helper functions

def chat(
    prompt,
    system_instruction=None,
    temperature=1.0,
    stop_sequences=None,
    tools=None,
    max_retries=5,
):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        stop_sequences=stop_sequences,
        tools=tools,
    )
    
    response = generate_content_with_retry(
        client=client,
        model=MODEL_ID,
        contents=prompt,
        config=config,
        max_attempts=max_retries,
    )
    return response

## Fire Risk Assessment Prompt

This detailed prompt instructs the model to perform a multi-step analysis of a property's fire risk based on residence location, tree overhang, and defensible space.

In [11]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

## Image Loading and Analysis

Finally, we load the satellite image using PIL and send it along with our prompt to Gemini for analysis. By passing them together in a list, the SDK automatically handles the multi-modal request.

In [12]:
# Read image data, feed into Gemini
image_path = "../../assets/images/prop7.png"
img = Image.open(image_path)

response = chat([img, prompt])
print(response.text)

Here is the analysis of the attached satellite image:

1.  **Residence identification:** The primary residence is a multi-sectioned, grey-roofed structure located centrally within the dense surrounding forest, with no clear property boundaries visible but appearing deeply embedded in natural vegetation.
2.  **Tree overhang analysis:** A significant portion, estimated to be well over 50% of the residence's roof area, is covered by dense tree canopy, particularly on the left and central sections of the structure.
3.  **Fire risk assessment:** The extensive overhanging and adjacent tree canopy creates numerous potential ember catch points and continuous fuel paths directly to the structure, bridging wildland vegetation to the roof.
4.  **Defensible space identification:** The trees form a continuous and dense canopy directly over and immediately surrounding the home, exhibiting obvious fuel ladders with vegetation extending from the ground level to the tree canopies adjacent to the struct